# Comprensió de les dades

Realitza els passos necessaris per entendre millor les dades.

Ometem el pas de la recollida de dades perquè et proporciono totes les dades necessàries.

## Imports

In [ ]:
# Importa les biblioteques, funcions, objectes... necessaris

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## Carrega el dataset

In [ ]:
df = pd.read_csv('../data/house_pricing/house_pricing.csv')

## Descriu les dades

Fes una primera inspecció bàsica de les dades: dimensions, primeres files, tipus de columnes, etc.

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

La columna YearBuilt és de tipus float. Hi ha algun valor nul o algun decimal diferent de 0?

In [ ]:
# Amb el mòdul 1 mirem si hi ha valors decimals (en tal cas el mòdul seria decimal)
df['YearBuilt'].isna().any() or (df['YearBuilt'] % 1 != 0).any()

Com que no és el cas, podem convertir-la a int.

In [ ]:
df['YearBuilt'] = df['YearBuilt'].astype(int)

El mateix per a `GarageYrBlt` i `SalePrice`, però s'han de mantenir com a float pels NaNs (que són float en numpy).

In [ ]:
(df['GarageYrBlt'].dropna() % 1 != 0).any()

In [ ]:
(df['SalePrice'].dropna() % 1 != 0).any()

## Anàlisi exploratòria de dades (EDA)

Resumeix les principals característiques de les dades amb visualitzacions.

### Data Profiling automàtic

In [ ]:
from data_profiling import ProfileReport

In [ ]:
profile = ProfileReport(
    df,
    title="House Pricing Report",
    explorative=True,  # Manté actives les anàlisis útils de text/tipus
    interactions={
        "continuous": False  # Desactiva la graella massiva de scatter plots per parells
    },
    correlations={
        "pearson": {
            "calculate": True,
            "threshold": 0.85
        },
        # Desactiva completament les alertes de mètriques de fons automàtiques
        "auto": {"calculate": False, "warn_high_correlations": False},
    },
)

In [ ]:
profile.to_file('../reports/house_pricing_report.html')

### Distribucions de les característiques

Comprova les distribucions de les característiques numèriques (histograma o boxplot) i de les característiques categòriques (gràfic de barres).

Només 1 pool area diferent de 0, no és una característica útil. Podem eliminar aquesta columna durant el processament.

In [ ]:
df['PoolArea'].value_counts()

El mateix passa amb `Utilities`:

In [ ]:
df['Utilities'].value_counts()

Hi havia una acumulació estranya a l'histograma de YearBuilt amb valors >= 2000. En comprovar-ho amb value_counts, tot sembla correcte.

In [ ]:
df[df['YearBuilt'] >= 2000]['YearBuilt'].value_counts()

Algunes de les variables categòriques tenen molt poques categories.

Després de revisar-les (codi a sota), les conclusions són:

- `MSZoning` només té dos comercials... La resta estan més o menys bé.
- `LandSlope` només té uns pocs exemples de Sev slope. Probablement no té cap efecte, però la podem mantenir.
- `HouseStyle` només té 2 2.5Fin i 5 2.5Unf. Com que SLvl i SFoyer són molt similars a 1.5Fin acabat, els podem incloure allà.
- `RoofStyle` sembla útil, però només té 3 de Flat i Gambrel, que és molt poc.
- `Foundation` sembla bé tot i que té Stone amb només 4 instàncies.
- `Street` té majoritàriament Pave però també alguns Grvl. Com que tenir Grvl podria ser un indicador d'un preu més baix, la podem mantenir.
- `Heating` té majoritàriament valors GasA. El segon més freqüent és GasW amb 10... que també és gas. Probablement podem eliminar aquesta característica.
- `PavedDrive` té majoritàriament valors Y, però també alguns dels altres. Deixa'l com està de moment.
- `Electrical` té FuseF i FuseP amb molt pocs exemples. Probablement podríem crear una característica booleana que indiqui `SBrkr` o `Fuse` (amb A, F i P junts), tot i que hi ha una gran diferència entre FuseA i FuseF (potència limitada) i FuseP (risc d'incendi). Potser podem crear una categoria FuseFP que combini els dos fusibles dolents. Comprovarem els NaNs més endavant.
- `MiscFeature` només té 1 exemple de TenC i Othr, aquestes categories no són útils. Podem crear una nova columna `HasShed` en lloc d'aquesta característica.

In [ ]:
to_check = [
    'MSZoning', 'LandSlope', 'HouseStyle', 'RoofStyle', 'Foundation', 'GarageType',
    'Street', 'Heating', 'PavedDrive', 'Electrical', 'MiscFeature'
]

for col in to_check:
    print(df[col].value_counts(dropna=False))
    print()

### Correlació entre característiques

L'única correlació alta és entre `GarageArea` i `GarageCars`, com era d'esperar. No obstant això, com que la correlació no és molt molt alta (0.87) i no tenim moltíssimes característiques, podem deixar-les totes dues.

In [ ]:
corrs = df.corr(numeric_only=True)

In [ ]:
corrs.loc['GarageCars', 'GarageArea']

### Pairplot

Visualitza la relació entre les diferents característiques amb un pairplot.

Sembla que hi ha una correlació no lineal entre `SalesPrice` i `YearBuilt`. Els models que funcionaran millor poden ser els no lineals.

In [ ]:
fts2pair = ['LotArea', 'GrLivArea', 'YearBuilt', 'GarageArea', 'SalePrice']

In [ ]:
sns.pairplot(df[fts2pair])
plt.show()

## Verifica la qualitat de les dades

- Són correctes les dades? Conté alguna columna errors importants que s'haurien de corregir?
- I els valors absents?
    - Com es representen (p. ex., cadena buida, NaN, -1, etc.)?
    - On apareixen?
    - Com de comuns són?
- No canviïs les dades, només inspecciona on apareixen aquests problemes.

### Nuls

Conclusions després de l'anàlisi següent:

- El nul de `Electrical` probablement hauria de ser SBrkr.
- El nul de `GarageType` significa que no hi ha garatge.
- El nul de `GarageYrBlt` significa que no hi ha garatge.
- El nul de `GarageFinish` significa que no hi ha garatge.
- El nul de `MiscFeature` significa que no s'ha d'afegir res (descripció de les dades). Podem calcular una nova característica `HasShed` i eliminar aquesta.
- Els nuls de `SalePrice` corresponen a preus del leaderboard que no coneixem. Estan bé.
- `Electtrical` són tots nuls. Ni tan sols està definida a la descripció de les dades. Probablement és una característica errònia que podem eliminar.

In [ ]:
# Proporció de nuls
prop_nans = df.isna().sum(axis=0) / len(df)
prop_nans[prop_nans > 0]

És sospitós que totes les característiques del garatge tinguin la mateixa proporció de nuls.

Segons el codi de sota, totes tres funcionen conjuntament: quan una és nul·la, també ho són les altres.

Com que a la descripció s'especifica que un `GarageFinish` nul significa que no hi ha garatge, sabem que un garatge NaN significa que no hi ha garatge.

In [ ]:
# Índexs de valors nuls de GarageType
set(df[df['GarageType'].isna()].index) == set(df[df['GarageYrBlt'].isna()].index) == set(df[df['GarageFinish'].isna()].index)

Per a `Electrical`, si dibuixem `YearBuilt` per a cada categoria elèctrica, veiem que les cases més noves totes tenen el tipus SBrkr. Tots els NaNs són cases noves, així que probablement són SBrkr.

In [ ]:
df.boxplot(column='YearBuilt', by='Electrical', grid=False);

In [ ]:
df[df['Electrical'].isna()]['YearBuilt'].describe()

Tots els nuls de `SalePrice` són del leaderboard, com s'esperava. Aquests estan bé.

In [ ]:
df[df['SalePrice'].isna()]['Split'].value_counts()

Molts NaNs a `MiscFeature`. Segons la descripció de les dades, això significa que no hi ha cap característica miscel·lània. Només un Othr i un TenC, no es pot aprendre res d'això. El que podem fer és canviar aquesta característica per una característica `HasShed`.

In [ ]:
df['MiscFeature'].value_counts(dropna=False)

### Duplicats

Cap Id duplicat (bé). Alguns duplicats després d'eliminar `Split`, `Id` i `SalePrice`. Tot i ser iguals, tenen un preu de venda diferent. Probablement podem assumir que són cases del mateix barri o similars. Encara que fossin errors (la mateixa casa afegida dues vegades amb preus diferents), com que no n'hi ha gaires, el seu impacte és mínim. Deixem-les.

In [ ]:
df['Id'].duplicated().any()

In [ ]:
# No hi ha duplicats quan eliminem Split i Id
df.drop(columns=['Split', 'Id']).duplicated().any()

In [ ]:
# Hi ha duplicats després d'eliminar aquestes columnes?
df.drop(columns=['Split', 'Id', 'SalePrice']).duplicated().any()

In [ ]:
df[df.drop(columns=['Split', 'Id', 'SalePrice']).duplicated(keep=False)].sort_values('LotArea')